In [ ]:
import sys
import os
import os
import phydrus as ps
import numpy as np
import warnings
import pandas as pd 
import os
import matplotlib.pyplot as plt
warnings.simplefilter(action='ignore', category=FutureWarning)
%matplotlib inline
current_dir = os.getcwd()

parent_dir = os.path.dirname(current_dir)
sys.path.append(parent_dir)

from model_builder import InfiltrationModel

In [ ]:

pond_max= 3.0
#flux = np.zeros(22, dtype=np.float64)
discretize = {"layers": [49, 51], "dz": [1.0, 1.0]}
flux = np.array([ -6.,-6.,-10,-10,-10,-2.,-2.,0.,0.,0.,1.,1., 1.,1.,1.,1.,1.,1.,1.,1.]) / 1440 
temp_time  = 1440.0
sim_time = temp_time * flux.shape[0]
trans = np.array([0.0] * (flux.shape[0]),dtype=np.float32) # if root wateruptake is active transpiration needs to be given 
bound_bot = 'free drainage'
soil_props = {"tr":[0.065, 0.095],"ths":[0.41, 0.41],"a":[0.075, 0.019],"n": [1.89, 1.31], "m":[1 - (1 / 1.89),1 - (1 / 1.31)],"ks":[106.1 / 1440, 6.24 / 1440], 
"L":[0.5, 0.5]}
hyd_model = 'VGM-AE'
model = InfiltrationModel(sim_time,temp_time,discretize,np.float32) #create the model!


hini = np.ones(model.nodes,dtype=np.float32) * -100.0
model.set_soil_model(hyd_model,soil_props)
model.set_boundary_conditions(bound_bot)
hout,sout,sink_out= model.set_run_solver(hini,flux,trans,pond_max,time_interval=1)


In [ ]:

# Folder for Hydrus files to be stored
ws = "example_2"
# Path to folder containing hydrus.exe 
exe = os.path.join('/Users/onursahin/Flux1Dpy/hydrus_source/source_code/source/hydrus')
# Description
desc = "Infiltration of Water into a Two-Layered Soil Profile"
# Create model
ml = ps.Model(exe_name=exe, ws_name=ws, name="model", description=desc, 
              mass_units="mmol", time_unit="days", length_unit="cm")
ml.basic_info["lFlux"] = True
ml.basic_info["lShort"] = False

time = np.arange(1,21)

ml.add_time_info(tmax=20, print_array=time, dt=0.001, dtmax=0.1)
ml.add_waterflow(model=3, top_bc=2, bot_bc=4,tolh=1,tolth=10**-3)
m = ml.get_empty_material_df(n=2)
m.loc[[1, 2]] = [[0.095, 0.41, 0.019, 1.31, 6.24, 0.5],
                 [0.065, 0.41, 0.075, 1.89, 106.1, 0.5]]
ml.add_material(m)
nodes = 100  # Disctretize soil column into n elements
depth = [-51, -100]  # Depth of the soil column
ihead = -100.0  # Determine initial Pressure Head
# Create Profile
profile = ps.create_profile(bot=depth, dx=abs(depth[-1] / nodes), h=ihead, mat=m.index)
ml.add_profile(profile)  # Add the profile
# Add observation nodes at depth
ml.add_obs_nodes([0, -50, -100])
time = (2, 5, 7, 10, 20)
bc = {"tAtm": time, "Prec": (6, 10, 2, 0, 0), "rSoil": (0, 0, 0, 0, 1)}
#bc = {"tAtm": time, "Prec": (0, 0, 0, 0, 0), "rSoil": (0.1, 0.1, 0.1, 0.1, 0.1)}
atm = pd.DataFrame(bc, index=time)

ml.add_atmospheric_bc(atm, hcrits=3.0, hcrita=50000)
ml.write_input()
ml.simulate()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import warnings

warnings.simplefilter(action='ignore', category=FutureWarning)

# Read the HYDRUS node information (profile snapshots)
hydrus_res = ml.read_nod_inf()
hydrus_times = sorted(list(hydrus_res.keys()))[0:10]

# Initialize lists to store the time series data
times = []
h_hydrus_top, h_hydrus_mid, h_hydrus_bot = [], [], []
h_custom_top, h_custom_mid, h_custom_bot = [], [], []

num_custom_nodes = hout.shape[1]

# Loop through all time steps to build the time series arrays
for t in hydrus_times:
    times.append(t)
    df_t = hydrus_res[t]
    
    # --- HYDRUS EXTRACTION ---
    # In HYDRUS nod_inf, row 0 is the surface, row -1 is the bottom
    mid_idx_hydrus = len(df_t) // 2
    h_hydrus_top.append(df_t['Head'].iloc[0])
    h_hydrus_mid.append(df_t['Head'].iloc[mid_idx_hydrus])
    h_hydrus_bot.append(df_t['Head'].iloc[-1])
    
    # --- CUSTOM MODEL EXTRACTION ---
    # Using your int(t) mapping to get the correct row in hout
    row_idx = int(t) 
    
    # In custom model, index -1 is the surface, index 0 is the bottom
    mid_idx_custom = num_custom_nodes // 2
    h_custom_top.append(hout[row_idx, -1])
    h_custom_mid.append(hout[row_idx, mid_idx_custom-1])
    h_custom_bot.append(hout[row_idx, 0])

# --- PLOTTING ---
fig, ax = plt.subplots(figsize=(8, 5))

# Plot HYDRUS (Solid lines with markers)
ax.plot(times, h_hydrus_top, color='blue', linestyle='-', marker='o', markersize=4, markevery=5, label='Hydrus - Top')
ax.plot(times, h_hydrus_mid, color='green', linestyle='-', marker='o', markersize=4, markevery=5, label='Hydrus - Middle')
ax.plot(times, h_hydrus_bot, color='red', linestyle='-', marker='o', markersize=4, markevery=5, label='Hydrus - Bottom')

# Plot Custom Model (Dashed lines matching colors)
ax.plot(times, h_custom_top, color='blue', linestyle='--', linewidth=2, label='Custom - Top',alpha=0.7)
ax.plot(times, h_custom_mid, color='green', linestyle='--', linewidth=2, label='Custom - Middle',alpha = 0.5)
ax.plot(times, h_custom_bot, color='red', linestyle='--', linewidth=2, label='Custom - Bottom',alpha = 0.7)

ax.set_xlabel('Time')
ax.set_ylabel('Pressure Head (h)')
ax.set_title('Time Series Comparison: HYDRUS vs Custom Model')

# Place legend outside the plot
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()
   

In [ ]:
hydrus_res = ml.read_nod_inf()
hydrus_times = sorted(list(hydrus_res.keys()))
print(hydrus_times)
for t in hydrus_times:
    plt.figure()
    df_t = hydrus_res[t]
    
    plt.plot(df_t['Moisture'],df_t['Depth'])
    plt.plot(sout[int(t),::-1],df_t['Depth'])
    plt.title(f'Time t = {t}')
    plt.show()
